# MAIA — Modelagem Ambiental com PCA

Este notebook implementa o modelo PCA ambiental do MAIA.

Fluxo:

`TB_ML_AMBIENTAL → features → StandardScaler → PCA → distância ao padrão histórico → score 0–100`

O score mede proximidade ao padrão histórico aprendido. Não representa probabilidade de falha, atolamento ou sinistro.


## 0 — Dependências

No `venv`:

```powershell
python -m pip install numpy pandas scikit-learn joblib matplotlib oracledb
```


In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CURRENT_PATH = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_PATH

for candidato in [CURRENT_PATH, *CURRENT_PATH.parents]:
    if (candidato / ".git").exists() or (candidato / ".env").exists():
        PROJECT_ROOT = candidato
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.executable)


## 1 — Importar o modelo PCA


In [ ]:
candidate_paths = [
    PROJECT_ROOT / "machine_learning",
    PROJECT_ROOT,
]

for p in candidate_paths:
    if (p / "maia_environmental_pca_model.py").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from maia_environmental_pca_model import EnvironmentalPCAModel

print("EnvironmentalPCAModel importado.")


## 2 — Ler a base limpa do Oracle


In [ ]:
from app.core.config_db import configurar_banco
from app.core.database import db

configurar_banco()

conn = db.get_connection()
cursor = conn.cursor()

try:
    cursor.execute(
        """
        SELECT
            DATA_OBSERVACAO,
            LATITUDE,
            LONGITUDE,
            FEATURES_JSON
        FROM TB_ML_AMBIENTAL
        WHERE FONTE = 'EDA_AMBIENTAL'
        ORDER BY DATA_OBSERVACAO
        """
    )

    rows = cursor.fetchall()

finally:
    cursor.close()
    db.release_connection(conn)

print(
    "Registros Oracle:",
    len(rows)
)

if not rows:
    raise ValueError(
        "TB_ML_AMBIENTAL não possui registros EDA_AMBIENTAL."
    )


In [ ]:
registros = []

for data_obs, latitude, longitude, features_json in rows:

    if hasattr(features_json, "read"):
        features_json = features_json.read()

    dados = json.loads(
        features_json
    )

    dados[
        "data_observacao_oracle"
    ] = data_obs

    dados[
        "latitude_oracle"
    ] = latitude

    dados[
        "longitude_oracle"
    ] = longitude

    registros.append(
        dados
    )

df_modelagem = pd.DataFrame(
    registros
)

df_modelagem.columns = [
    str(c)
    .strip()
    .lower()
    .replace(" ", "_")

    for c
    in df_modelagem.columns
]

print(
    "Shape:",
    df_modelagem.shape
)

display(
    df_modelagem.head()
)


## 3 — Ajustar o PCA

O modelo remove features constantes automaticamente e usa `StandardScaler` antes do PCA.


In [ ]:
modelo_pca = EnvironmentalPCAModel(
    explained_variance_target=0.90,
    min_components=2,
)

modelo_pca.fit(
    df_modelagem
)

print(
    "Linhas usadas:",
    modelo_pca.fit_summary_.n_rows
)

print(
    "Features usadas:",
    modelo_pca.fit_summary_.n_features
)

print(
    "Componentes retidos:",
    modelo_pca.fit_summary_.n_components
)

print(
    "Constantes removidas:",
    modelo_pca.constant_features_
)


## 4 — Variância explicada


In [ ]:
variancia = modelo_pca.explained_variance()

display(
    variancia
)

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    variancia[
        "component"
    ],
    variancia[
        "cumulative_explained_variance"
    ],
    marker="o",
)

plt.axhline(
    0.90,
    linewidth=1
)

plt.ylim(
    0,
    1.05
)

plt.ylabel(
    "Variância explicada acumulada"
)

plt.xlabel(
    "Componente"
)

plt.title(
    "PCA ambiental — variância explicada"
)

plt.tight_layout()
plt.show()


## 5 — Loadings


In [ ]:
loadings = modelo_pca.loadings()

display(
    loadings
)

display(
    modelo_pca.top_loadings(
        n=8
    )
)


## 6 — Gerar score relativo de condição ambiental


In [ ]:
resultado = modelo_pca.transform(
    df_modelagem
)

display(
    resultado.tail(15)
)


## 7 — Salvar modelo treinado


In [ ]:
ARTIFACT_DIR = (
    PROJECT_ROOT
    / "machine_learning"
    / "artifacts"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_PATH = (
    ARTIFACT_DIR
    / "environmental_pca_model.joblib"
)

modelo_pca.save(
    MODEL_PATH
)

print(
    "Modelo salvo em:",
    MODEL_PATH
)


# 8 — TESTE FINAL


In [ ]:
observacao_teste = (
    df_modelagem
    .iloc[-1]
    .to_dict()
)

teste = modelo_pca.predict_one(
    observacao_teste
)

print("=" * 64)
print("MAIA — TESTE DO MODELO AMBIENTAL PCA")
print("=" * 64)

print(
    "Score:",
    f"{teste['score_condicao_ambiental']:.2f}/100"
)

print(
    "Atipicidade:",
    f"{teste['indice_atipicidade_pct']:.2f}%"
)

print(
    "Faixa:",
    teste["faixa_condicao"]
)

print(
    "Distância PCA:",
    f"{teste['pca_distance']:.4f}"
)

print()

for i in range(
    1,
    modelo_pca.pca_.n_components_ + 1
):
    print(
        f"PC{i}:",
        f"{teste[f'PC{i}']:.4f}"
    )


## 9 — Testar recarga do artefato


In [ ]:
modelo_recarregado = (
    EnvironmentalPCAModel.load(
        MODEL_PATH
    )
)

teste_recarregado = (
    modelo_recarregado.predict_one(
        observacao_teste
    )
)

assert abs(
    teste_recarregado[
        "score_condicao_ambiental"
    ]
    - teste[
        "score_condicao_ambiental"
    ]
) < 1e-9

print(
    "OK — modelo salvo e recarregado."
)
